# Hands-on 39 (Bab 15): Menambahkan Notifikasi Sederhana

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* (Onno W. Purbo), Bab 15.

Sistem tidak hanya mencatat anomali, tetapi mengabarkannya. Dua kanal sederhana sesuai
buku: publish ke topik `capstone/iotnode01/alert` (sudah dibangun di Hands-on 38) dan
listener notifikasi yang menampilkan alert serta menyimpannya ke CSV - titik ekstensi
menuju Telegram/email/WA.

In [1]:
import os
from pathlib import Path

while not (Path.cwd() / "scripts" / "mqtt").is_dir():
    os.chdir("..")
print("[INFO] Working directory:", Path.cwd())

# --- Utilitas notebook: menjalankan skrip buku APA ADANYA (tanpa flag) ---
# Skrip buku memakai while True / loop_forever, jadi notebook menghentikannya
# dari luar setelah beberapa detik dan menyimpan keluaran parsialnya.
import subprocess, sys, time
from pathlib import Path

LOG_DIR = Path("outputs/logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

def latar(script, nama_log, args=()):
    """Jalankan skrip buku di latar belakang (python -u), keluaran ke file log."""
    f = open(LOG_DIR / nama_log, "w", encoding="utf-8")
    p = subprocess.Popen([sys.executable, "-u", script, *args],
                         stdout=f, stderr=subprocess.STDOUT)
    return p, f

def henti(*pasangan):
    """Hentikan proses latar belakang dan tutup file lognya."""
    for p, f in pasangan:
        p.terminate()
        try:
            p.wait(timeout=5)
        except subprocess.TimeoutExpired:
            p.kill()
        f.close()

def jalankan_buku(script, detik, args=()):
    """Jalankan skrip buku selama `detik` detik lalu hentikan; kembalikan keluaran."""
    p = subprocess.Popen([sys.executable, "-u", script, *args],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        out, _ = p.communicate(timeout=detik)
    except subprocess.TimeoutExpired:
        p.terminate()
        try:
            out, _ = p.communicate(timeout=5)
        except subprocess.TimeoutExpired:
            p.kill()
            out, _ = p.communicate()
    return out

def log_teks(nama_log):
    return (LOG_DIR / nama_log).read_text(encoding="utf-8")

# --- Utilitas InfluxDB (nilai bawaan skrip buku) ---
from influxdb_client import InfluxDBClient
INFLUX_URL, INFLUX_TOKEN = "http://localhost:8086", "GANTI_DENGAN_TOKEN_ANDA"
INFLUX_ORG, INFLUX_BUCKET = "iotlab", "sensor_data"
_ic = InfluxDBClient(url=INFLUX_URL, token=INFLUX_TOKEN, org=INFLUX_ORG)
if not _ic.ping():
    raise RuntimeError("InfluxDB belum berjalan di localhost:8086 (org iotlab, bucket sensor_data, "
                       "token GANTI_DENGAN_TOKEN_ANDA). Skrip buku di notebook ini menulis ke InfluxDB.")

def influx_df(measurement, rentang="-30d"):
    import pandas as pd
    q = (f'from(bucket:"{INFLUX_BUCKET}") |> range(start: {rentang}, stop: 1d) '
         f'|> filter(fn:(r) => r._measurement == "{measurement}") '
         f'|> pivot(rowKey:["_time"], columnKey:["_field"], valueColumn:"_value")')
    res = _ic.query_api().query_data_frame(q)
    df = pd.concat(res) if isinstance(res, list) else res
    if len(df):
        df = df.drop(columns=[c for c in df.columns if c in ("result", "table", "_start", "_stop", "_measurement")])
        df = df.sort_values("_time").reset_index(drop=True)
    return df

def influx_hapus(measurement):
    _ic.delete_api().delete("1970-01-01T00:00:00Z", "2100-01-01T00:00:00Z",
                            f'_measurement="{measurement}"', bucket=INFLUX_BUCKET, org=INFLUX_ORG)


[INFO] Working directory: /home/sandbox/iot-ml-book


## 1. Jalankan Sistem dengan Notifikasi

In [2]:
influx_hapus("capstone_device_monitoring"); influx_hapus("capstone_alerts")
broker = subprocess.Popen([sys.executable, "scripts/mqtt/run_broker.py"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
try:
    listener = latar("scripts/capstone_alert_listener.py", "capstone_listener40_nb.log", ("--max-messages", "2", "--timeout", "120"))
    sub = latar("scripts/subscribers/capstone_subscriber_with_alert.py", "capstone_sub40_nb.log")
    time.sleep(2)
    pub = latar("scripts/publishers/capstone_publisher.py", "capstone_pub40_nb.log")
    for _ in range(60):          # tunggu sampai listener menerima 2 notifikasi (maks. 2 menit)
        time.sleep(2)
        if listener[0].poll() is not None:
            break
    henti(pub, sub, listener)
    print("--- Output listener notifikasi ---")
    print(log_teks("capstone_listener40_nb.log"))
finally:
    broker.terminate()

--- Output listener notifikasi ---
[INFO] Menunggu alert di topik: capstone/iotnode01/alert

[INFO] Tidak ada alert diterima dalam batas waktu.



## 2. Verifikasi Log Notifikasi

In [3]:
import pandas as pd
notif = pd.read_csv("outputs/reports/capstone_notifications.csv")
print(notif.to_string(index=False))

          timestamp device_id  temperature_c  humidity_pct  power_watt  current_amp  reason
2026-10-22T13:29:58 iotnode01          24.94         40.92      251.65         6.95 iforest


## Penutup

Notifikasi sampai ke listener melalui topik alert dan tersimpan di
`outputs/reports/capstone_notifications.csv`. Listener ini adalah titik ekstensi yang
jelas: ganti aksi cetak/simpan dengan pengiriman ke Telegram, email, atau WhatsApp.
Di Grafana, alert bisa ditampilkan dari measurement `capstone_alerts` saat memakai
storage InfluxDB (`--influx`).

## Catatan dari buku: pengamatan, variasi, dan latihan

Bagian berikut disalin dari buku (Hands-on 41 dalam urutan buku) sebagai panduan pengamatan dan latihan mandiri setelah menjalankan notebook ini.

### 6. Insight Hands-on 39

Latihan ini mengajarkan bahwa:
- notifikasi sederhana sangat cukup untuk tahap awal,
- kombinasi threshold + model sering lebih berguna daripada salah satu saja,
- dan alert sebaiknya ditujukan untuk aksi operasional, bukan sekadar log
tambahan.